Dataset


In [2]:
import kagglehub
path = kagglehub.dataset_download("andradaolteanu/gtzan-dataset-music-genre-classification")
print("Path to dataset files:", path)

Using Colab cache for faster access to the 'gtzan-dataset-music-genre-classification' dataset.
Path to dataset files: /kaggle/input/gtzan-dataset-music-genre-classification


In [3]:
import librosa
import numpy as np
from pathlib import Path
import random

import time
from sklearn.metrics import confusion_matrix

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
fft_total_time = 0  # Time to make a spectrogram for each song (10 total songs)
inference_total_time = 0
e2e_time = 0
correct_predictions = 0
total_predictions = 0
def my_spectrogram(x, m, fs):

    # Inputs:
    # x -- data vector
    # m -- block size, must be even
    # fs -- sampling rate

    if (m % 2 != 0):
        raise ValueError("Choose an even block size")

    # Pad x up to a multiple of m
    lx = len(x)
    nt = int(np.ceil(lx/m))
    # plus 1 is so that the overlapping window works
    xp = np.zeros((nt + 1) * m)
    xp[:lx] = x


    # Rectangular windowing
    # use reshape to make it an m by nt matrix
    xm = xp[:nt * m].reshape((nt, m)).T

    # 50% overlapping blocks
    n1 = m // 2
    n2 = n1+nt*m

    xmi = xp[n1:n2].reshape((nt, m)).T

    xm3 = np.zeros((m,2*nt))
    xm3[:, 0::2] = xm
    xm3[:, 1::2] = xmi

    window = np.hamming(m)
    xm3 *= window[:, None]

    xmf3 = np.fft.fft(xm3, axis=0)

    spectrogram = np.abs(xmf3[:m // 2, :])
    spectrogram = 20 * np.log10(
    np.abs(xmf3[:m // 2, :]) + 1e-10
    )
    # Axes
    freqs = np.arange(0, m // 2) * fs / m
    times = np.arange(1, 2 * nt + 1) * m / (2 * fs)

    return spectrogram, freqs, times

####################### SONG LOADING ########################



In [4]:
import soundfile as sf

def is_valid_audio(path):
    try:
        sf.info(str(path))
        return True
    except:
        return False

In [5]:
from pathlib import Path
import numpy as np
import librosa
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

songs_folder = Path(f"{path}/Data/genres_original")

chunk_size = 50
sr = 22050
fft_size = 1024

genres = sorted([d.name for d in songs_folder.iterdir() if d.is_dir()])
genre_to_label = {g: i for i, g in enumerate(genres)}

index = []

for genre in genres:
    genre_folder = songs_folder / genre
    label = genre_to_label[genre]

    count = 0
    for song_path in genre_folder.glob("*.wav"):
        # count += 1
        # if count > 5:
        #     break
        if is_valid_audio(song_path):
            index.append((song_path, label))
        else:
            print(f"Skipping corrupted file: {song_path}")



train_index, test_index = train_test_split(
    index,
    test_size=0.2,
    random_state=42,
    stratify=[y for _, y in index]
)




Skipping corrupted file: /kaggle/input/gtzan-dataset-music-genre-classification/Data/genres_original/jazz/jazz.00054.wav


In [6]:
class SmallAudioDataset(Dataset):
    def __init__(self, index, cache, chunk_size, train=True, chunks_per_song=50):
        self.index = index
        self.cache = cache
        self.chunk_size = chunk_size
        self.train = train
        self.chunks_per_song = chunks_per_song

    def __len__(self):
        return len(self.index) * self.chunks_per_song

    def __getitem__(self, idx):
        song_path, label = self.index[idx // self.chunks_per_song]
        spec = self.cache[str(song_path)]  # already computed

        max_start = spec.shape[0] - self.chunk_size
        start = np.random.randint(0, max_start) if self.train else max_start // 2
        chunk = spec[start:start + self.chunk_size]

        return (
            torch.tensor(chunk, dtype=torch.float32),
            torch.tensor(label, dtype=torch.long)
        )

In [7]:
# Run this ONCE before training
from tqdm import tqdm

def precompute(index, sr=22050, fft_size=1024):
    cache = {}
    for song_path, label in tqdm(index):
        audio, _ = librosa.load(song_path, sr=sr)
        spec, _, _ = my_spectrogram(audio, m=fft_size, fs=sr)
        spec = spec.T
        spec = (spec - spec.mean()) / (spec.std() + 1e-8)
        cache[str(song_path)] = spec
    return cache

print("Precomputing...")
train_cache = precompute(train_index)
test_cache  = precompute(test_index)

# Recreate datasets with cache
train_dataset = SmallAudioDataset(train_index, train_cache, chunk_size=50, train=True)
test_dataset  = SmallAudioDataset(test_index,  test_cache,  chunk_size=50, train=False)
train_loader  = DataLoader(train_dataset, batch_size=64, shuffle=True,  num_workers=0)
test_loader   = DataLoader(test_dataset,  batch_size=64, shuffle=False, num_workers=0)

Precomputing...


100%|██████████| 200/200 [00:16<00:00, 11.98it/s]


In [8]:
class SmallLSTM(nn.Module):
    def __init__(self):
        super().__init__()
        self.proj = nn.Linear(512, 128)
        self.lstm = nn.LSTM(
            input_size=128,
            hidden_size=128,
            num_layers=2,
            batch_first=True,
            dropout=0.2
        )
        self.fc = nn.Linear(128, 10)

    def forward(self, x):
        x = self.proj(x)
        x, _ = self.lstm(x)
        x = x[:, -1, :]
        x = self.fc(x)
        return x

In [9]:
def save_model(model, path):
    torch.save(model.state_dict(), path)

def load_model(model, path):
    model.load_state_dict(torch.load(path))
    return model

In [10]:
print(f"Training songs: {len(train_index)}")
print(f"Test songs: {len(test_index)}")
print(f"Train batches: {len(train_loader)}")
print(f"Test batches: {len(test_loader)}")
print(f"Train samples: {len(train_dataset)}")

Training songs: 799
Test songs: 200
Train batches: 625
Test batches: 157
Train samples: 39950


In [11]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = SmallLSTM().to(device)
x, y = next(iter(train_loader))
x, y = x.to(device), y.to(device)
print(x.shape)   # must be (64, 50, 512)
out = model(x)
print(out.shape) # must be (64, 10)
print(torch.nn.functional.cross_entropy(out, y).item())  # should be ~2.3

torch.Size([64, 50, 512])
torch.Size([64, 10])
2.304565906524658


In [12]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = SmallLSTM()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)  # Adam not AdamW
criterion = nn.CrossEntropyLoss()
epochs = 30

model.to(device)
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for batch_x, batch_y in train_loader:
        optimizer.zero_grad()
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)
        predictions = model(batch_x)
        loss = criterion(predictions, batch_y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # clip gradients
        optimizer.step()
        total_loss += loss.item()


    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

torch.save(model.state_dict(), "model_small.pth")

Epoch 1, Loss: 907.9217
Epoch 2, Loss: 623.9175
Epoch 3, Loss: 471.0261
Epoch 4, Loss: 359.9739
Epoch 5, Loss: 282.5355
Epoch 6, Loss: 226.3141
Epoch 7, Loss: 185.8108
Epoch 8, Loss: 164.1848
Epoch 9, Loss: 135.4033
Epoch 10, Loss: 120.0792
Epoch 11, Loss: 109.1458
Epoch 12, Loss: 96.9353
Epoch 13, Loss: 91.2013
Epoch 14, Loss: 83.2808
Epoch 15, Loss: 73.1554
Epoch 16, Loss: 69.0325
Epoch 17, Loss: 66.7191
Epoch 18, Loss: 58.0418
Epoch 19, Loss: 59.6561
Epoch 20, Loss: 52.9205
Epoch 21, Loss: 51.9715
Epoch 22, Loss: 48.3032
Epoch 23, Loss: 47.8710
Epoch 24, Loss: 44.0281
Epoch 25, Loss: 46.2464
Epoch 26, Loss: 40.9634
Epoch 27, Loss: 42.9008
Epoch 28, Loss: 44.0904
Epoch 29, Loss: 37.5609
Epoch 30, Loss: 40.5701


In [13]:
import torch
import numpy as np

def evaluate(model, loader, device="cpu"):
    model.eval()

    all_preds = []
    all_targets = []

    with torch.no_grad():

        for x, y in loader:

            x = x.to(device)
            y = y.to(device)

            logits = model(x)
            preds = torch.argmax(logits, dim=1)

            all_preds.append(preds.cpu().numpy())
            all_targets.append(y.cpu().numpy())

    all_preds = np.concatenate(all_preds)
    all_targets = np.concatenate(all_targets)

    accuracy = (all_preds == all_targets).mean() * 100

    return accuracy, all_preds, all_targets

In [14]:

test_acc, preds, labels = evaluate(model, test_loader, device)


print("Test accuracy:", test_acc)

Test accuracy: 66.5


In [16]:
import torch
import numpy as np

GENRES = ["blues", "classical", "country", "disco", "hiphop",
          "jazz", "metal", "pop", "reggae", "rock"]

def export(tensor, name, path):
    arr = tensor.detach().numpy().flatten()
    np.savetxt(f"{path}/{name}.csv", arr, delimiter=',')
    print(f"Exported {name}.csv — shape {tensor.shape}")

OUT = "/content/weights"
import os
os.makedirs(OUT, exist_ok=True)

model_export = SmallLSTM()
model_export.load_state_dict(torch.load('model_small.pth', map_location='cpu'))
model_export.eval()

export(model_export.lstm.weight_ih_l0, "weight_ih_l0", OUT)
export(model_export.lstm.weight_hh_l0, "weight_hh_l0", OUT)
export(model_export.lstm.bias_ih_l0,   "bias_ih_l0",   OUT)
export(model_export.lstm.bias_hh_l0,   "bias_hh_l0",   OUT)
export(model_export.lstm.weight_ih_l1, "weight_ih_l1", OUT)
export(model_export.lstm.weight_hh_l1, "weight_hh_l1", OUT)
export(model_export.lstm.bias_ih_l1,   "bias_ih_l1",   OUT)
export(model_export.lstm.bias_hh_l1,   "bias_hh_l1",   OUT)
export(model_export.fc.weight,         "fc_weight",     OUT)
export(model_export.fc.bias,           "fc_bias",       OUT)

# export one test input chunk
x_sample, y_sample = test_dataset[0]
proj_out = model_export.proj(x_sample.unsqueeze(0)).squeeze(0).detach().numpy()
np.savetxt(f"{OUT}/input_seq.csv", proj_out.flatten(), delimiter=',')
print(f"Exported input_seq.csv — shape {proj_out.shape}")
print(f"True label: {y_sample.item()} ({GENRES[y_sample.item()]})")

Exported weight_ih_l0.csv — shape torch.Size([512, 128])
Exported weight_hh_l0.csv — shape torch.Size([512, 128])
Exported bias_ih_l0.csv — shape torch.Size([512])
Exported bias_hh_l0.csv — shape torch.Size([512])
Exported weight_ih_l1.csv — shape torch.Size([512, 128])
Exported weight_hh_l1.csv — shape torch.Size([512, 128])
Exported bias_ih_l1.csv — shape torch.Size([512])
Exported bias_hh_l1.csv — shape torch.Size([512])
Exported fc_weight.csv — shape torch.Size([10, 128])
Exported fc_bias.csv — shape torch.Size([10])
Exported input_seq.csv — shape (50, 128)
True label: 1 (classical)


In [17]:
import shutil
shutil.make_archive("/content/weights", 'zip', OUT)
from google.colab import files
files.download("/content/weights.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>